# Credit Default Risk Modeling, Calibration and Decision Policy

**English objective:** Estimate next-month default probabilities, validate ranking stability, create operational risk tiers, and select a review threshold under an explicit business-cost assumption.

**中文目标：** 预测信用卡客户下月违约概率，验证模型排序稳定性，建立风险分层，并在明确业务成本假设下选择客户筛查阈值。

## Research design

The test set is reserved for final evaluation. Feature engineering, hyperparameter tuning, model selection, calibration, and threshold selection use training data only. This avoids test-set leakage.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from src.data_preprocessing import build_preprocessor, clean_credit_data, load_credit_default_data, split_features_target
from src.feature_engineering import add_business_features
from src.model_training import (calibrate_model, compare_feature_engineering_cv, cross_validate_models,
                                get_oof_probabilities, split_train_test, train_models, tune_hist_gradient_boosting)
from src.evaluation import (decile_lift_summary, evaluate_models, evaluate_probabilities,
                            permutation_feature_importance, risk_tier_summary, select_cost_optimal_threshold,
                            threshold_analysis)
from src.visualization import (plot_calibration_curve, plot_decile_default_rate, plot_lift_curve,
                               plot_permutation_importance, plot_precision_recall_curves, plot_risk_tier_summary,
                               plot_roc_curves, plot_threshold_cost, plot_threshold_tradeoff)

DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and clean data

In [ ]:
df = clean_credit_data(load_credit_default_data(DATA_DIR))
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
print('Portfolio default rate:', f"{df['default_next_month'].mean():.2%}")
display(df.head())

## 2. Exploratory analysis

In [ ]:
display(df['default_next_month'].value_counts().rename_axis('default').to_frame('clients'))
display(df.groupby('repay_status_sep')['default_next_month'].agg(['count', 'mean']).rename(columns={'mean': 'default_rate'}))
display(df.describe().T)

## 3. Business feature engineering

The domain transformer adds delinquency frequency and severity, utilisation, payment coverage, bill trend, and payment-gap features. In model training, this transformer is fitted inside each Pipeline.

In [ ]:
X, y = split_features_target(df)
featured_example = add_business_features(X.head())
print('Raw feature count:', X.shape[1])
print('Feature count after business engineering:', featured_example.shape[1])
display(featured_example.iloc[:, -22:].head())

## 4. Untouched holdout split

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(X, y)
print('Training rows:', len(X_train), '| Test rows:', len(X_test))
print('Training default rate:', f'{y_train.mean():.2%}', '| Test default rate:', f'{y_test.mean():.2%}')

## 5. Benchmark training and hyperparameter tuning

In [ ]:
preprocessor = build_preprocessor(include_engineered=True)
models = train_models(X_train, y_train, preprocessor)
tuned_model, search_results = tune_hist_gradient_boosting(X_train, y_train, preprocessor, n_iter=12, cv_folds=3)
models['Tuned HistGradientBoosting'] = tuned_model
display(search_results.head())

## 6. Cross-validation stability and feature-engineering ablation

A stable model should perform consistently across folds. The ablation study quantifies the contribution of business features.

In [ ]:
cv_metrics = cross_validate_models(models, X_train, y_train, cv_folds=5)
feature_impact = compare_feature_engineering_cv(X_train, y_train, cv_folds=5)
display(cv_metrics.style.format({c: '{:.4f}' for c in cv_metrics.columns if c not in ['model', 'cv_folds']}))
display(feature_impact.style.format({c: '{:.4f}' for c in feature_impact.columns if c not in ['model', 'cv_folds']}))

## 7. Training-only model selection and probability calibration

The ranking model is selected from cross-validation results rather than test-set performance. Raw and sigmoid-calibrated probabilities are compared using out-of-fold Brier score and log loss.

In [ ]:
best_name = cv_metrics.iloc[0]['model']
best_model = models[best_name]
calibrated_model = calibrate_model(best_model, X_train, y_train, cv_folds=3)
calibrated_name = f'Calibrated {best_name}'
reporting_models = {**models, calibrated_name: calibrated_model}
raw_oof_prob = get_oof_probabilities(best_model, X_train, y_train, cv_folds=3)
calibrated_oof_prob = get_oof_probabilities(calibrated_model, X_train, y_train, cv_folds=3)
calibration_comparison = pd.DataFrame([
    evaluate_probabilities(best_name, y_train, raw_oof_prob),
    evaluate_probabilities(calibrated_name, y_train, calibrated_oof_prob),
]).sort_values(['brier_score', 'log_loss'])
operational_name = calibration_comparison.iloc[0]['model']
operational_model = calibrated_model if operational_name == calibrated_name else best_model
oof_prob = calibrated_oof_prob if operational_name == calibrated_name else raw_oof_prob
print('CV-selected ranking model:', best_name)
print('OOF-selected probability model:', operational_name)
display(calibration_comparison[['model', 'brier_score', 'log_loss', 'roc_auc', 'average_precision']])

## 8. Out-of-fold threshold policy

The illustrative cost assumption treats a missed default as five times more costly than a false positive. The threshold is selected from out-of-fold training probabilities, not the test set.

In [ ]:
threshold_df = threshold_analysis(y_train, oof_prob, false_negative_cost=5, false_positive_cost=1)
selected_threshold = select_cost_optimal_threshold(threshold_df)
print('Selected threshold:', selected_threshold)
plot_threshold_tradeoff(threshold_df, selected_threshold, OUTPUT_DIR)
plot_threshold_cost(threshold_df, selected_threshold, OUTPUT_DIR)

## 9. Final untouched holdout evaluation

ROC-AUC and KS measure discrimination. Average Precision is informative under class imbalance. Brier score and log loss evaluate probability quality. Capture and lift quantify review efficiency.

In [ ]:
metrics_df = evaluate_models(reporting_models, X_test, y_test)
y_prob = operational_model.predict_proba(X_test)[:, 1]
threshold_comparison = pd.DataFrame([
    evaluate_probabilities(operational_name + ' | 0.50', y_test, y_prob, 0.50),
    evaluate_probabilities(operational_name + ' | cost optimised', y_test, y_prob, selected_threshold),
])
display(metrics_df.style.format({c: '{:.4f}' for c in metrics_df.columns if c != 'model'}))
display(threshold_comparison[['model', 'threshold', 'precision', 'recall', 'f1', 'accuracy']].style.format({c: '{:.4f}' for c in ['threshold', 'precision', 'recall', 'f1', 'accuracy']}))
plot_roc_curves(reporting_models, X_test, y_test, OUTPUT_DIR)
plot_precision_recall_curves(reporting_models, X_test, y_test, OUTPUT_DIR)
plot_calibration_curve(y_test, y_prob, operational_name, OUTPUT_DIR)

## 10. Risk tiers, lift, and default capture

In [ ]:
tier_summary = risk_tier_summary(y_test, y_prob)
decile_summary = decile_lift_summary(y_test, y_prob)
display(tier_summary.style.format({c: '{:.2%}' for c in ['share', 'actual_default_rate', 'avg_predicted_default_prob', 'min_predicted_default_prob', 'max_predicted_default_prob']}))
display(decile_summary.head())
plot_risk_tier_summary(tier_summary, OUTPUT_DIR)
plot_lift_curve(decile_summary, OUTPUT_DIR)
plot_decile_default_rate(decile_summary, OUTPUT_DIR)

## 11. Model explainability

Permutation importance measures the decrease in holdout ROC-AUC when each transformed feature is disrupted. It supports interpretation without claiming causality.

In [ ]:
importance_df = permutation_feature_importance(best_model, X_test, y_test, n_repeats=5)
display(importance_df.head(15))
plot_permutation_importance(importance_df, OUTPUT_DIR)

## 12. Business interpretation

The score is useful because it ranks and segments customers rather than only issuing a binary label. Recent delinquency and repayment behaviour are direct signals of payment discipline; utilisation, bill volatility, credit limit, and payment coverage provide evidence about financial pressure and repayment capacity.

该模型输出客户下月违约概率，并通过风险层、Lift 和捕获率支持差异化贷后管理。阈值应根据真实机构的漏报损失、人工审核成本和风险偏好重新设定，而不是机械使用 0.5。

## 13. Resume-ready Chinese bullets

- 基于 30,000 条信用卡客户数据构建端到端违约概率模型，在 sklearn Pipeline 内完成业务特征工程、交叉验证调参及概率校准，调优模型 5 折 ROC-AUC 达 0.7869 ± 0.0042，独立测试集 AUC 达 0.7821、KS 达 0.4312。
- 构建逾期频次、近期加权逾期程度、额度使用率、还款覆盖率和账单趋势等业务衍生特征，并通过 permutation importance 识别关键风险驱动因素。
- 基于训练集 out-of-fold 预测和非对称业务成本优化决策阈值，将独立测试集违约客户召回率由 36.62% 提升至 74.91%。
- 建立客户风险分层与 Lift/Capture 报告，最高风险 10% 客户覆盖 31.95% 的违约客户、Lift 达 3.20x，最高风险层实际违约率达 77.00%。